In [ ]:
# libraries
import numpy as np
import librosa
from scipy.signal import lfilter
from scipy.fft import fft
# from scipy.signal import triang
from scipy.fftpack import dct


In [ ]:
def deltas(x,w=9):
    nr, nc = x.shape
    
def trimf(x, params):
    """
    生成一个三角形滤波器。
    :param x: 定义滤波器响应的频率向量。
    :param params: 一个包含三个元素的列表或元组，定义了三角形的左端点、顶点和右端点。
    :return: 一个 numpy 数组，表示三角形滤波器的响应。
    """
    a, b, c = params
    assert a <= b <= c, "三角形的三个点必须满足 a <= b <= c"
    
    y = np.zeros_like(x)
    # 左侧上升沿
    left_indices = np.logical_and(a < x, x < b)
    y[left_indices] = (x[left_indices] - a) / (b - a)
    # 右侧下降沿
    right_indices = np.logical_and(b < x, x < c)
    y[right_indices] = (c - x[right_indices]) / (c - b)
    # 顶点
    y[x == b] = 1
    
    return y
def extract_lfcc(speech,Fs,Window_Length,NFFT,No_Filter):
    
    speech = lfilter([1, -0.97], 1, speech)
    frame_length_inSample = round((Fs / 1000) * Window_Length)
    hop_length_inSample = round(frame_length_inSample / 2.0)
    framedspeech = librosa.util.frame(speech, frame_length=frame_length_inSample, hop_length=hop_length_inSample).T
   
    # 假设 framedspeech, frame_length_inSample 已经定义
    w = np.hamming(frame_length_inSample)
    y_framed = framedspeech * w[:, np.newaxis]
    f = (Fs / 2) * np.linspace(0, 1, int(NFFT / 2) + 1)
    filbandwidthsf = np.linspace(np.min(f), np.max(f), No_Filter + 2)
    fr_all = (np.abs(fft(y_framed.T, NFFT)) ** 2)
    fa_all = fr_all[:int(NFFT / 2) + 1, :].T
    filterbank = np.zeros((int(NFFT / 2) + 1, No_Filter))
    for i in range(No_Filter):
        filterbank[:, i] = trimf(f, (filbandwidthsf[i], filbandwidthsf[i+1], filbandwidthsf[i+2]))

# 应用滤波器组
    eps_value = np.finfo(float).eps
    filbanksum = np.dot(fa_all, filterbank)
    t = dct(np.log10(filbanksum.T + eps_value), norm='ortho')
    t = t[:No_Filter, :]
    stat = t.T
    